In [1]:
import sys
from pathlib import Path
root_path = Path().resolve().parent
sys.path.insert(0, str(root_path))

from torch.utils.data import DataLoader, Subset
from models.vgg16 import BrainMRI, VGG16, get_train_transforms, get_val_transforms
import os
import json

import matplotlib.pyplot as plt
%matplotlib inline

In [71]:
# root_folder_dir = "../data/brain-tumor-mri-deduplicated/"
# split = 'train'

# train_transforms = get_train_transforms(do_augment=False)
# train_dataset = BrainMRI(root_folder_dir, split, train_transforms)
# train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# idx_to_classes = {idx: cls for idx, cls in enumerate(train_dataset.classes)}

In [72]:
# fig, axes = plt.subplots(2, 4, figsize=(12, 8))
# indices = torch.randperm(len(train_dataset))
# visualize = Subset(train_dataset, indices)

# mean = torch.Tensor([0.485, 0.456, 0.406])
# std = torch.Tensor([0.229, 0.224, 0.225])
# for i, sample in enumerate(visualize):
#     if i > 7:
#         break
#     vis = sample[0].permute(1, 2, 0)
#     vis = vis * std.view(1, 1, 3) + mean.view(1, 1, 3)
#     vis = torch.clamp(vis, min=0, max=1).numpy()

#     row = i // 4
#     col = i % 4
#     axes[row, col].imshow(vis)
#     axes[row, col].set_title(idx_to_classes[sample[1]])
#     axes[row, col].axis('off')

# plt.tight_layout()
# plt.show()

In [2]:
import tqdm
import torch
from torch.utils.data import DataLoader
from models.vgg16 import VGG16, BrainMRI, get_train_transforms, get_val_transforms
import torch.functional as F

In [3]:
def save_checkpoint(model, optimizer, lr_scheduler, state, OUT_DIR):
    checkpoint = {
        'epoch': state['epoch'],
        'global_step': state['global_step'],
        'epoch_step': state['epoch_step'],
        'best_val_loss': state['best_val_loss'],
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': lr_scheduler.state_dict(),
    }
    torch.save(checkpoint, OUT_DIR / "checkpoint.pth")
    print(f"Checkpoint saved at epoch {state['epoch']}")

In [4]:
def load_checkpoint(model, optimizer, lr_scheduler, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    lr_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    state = {
        'epoch': checkpoint['epoch'],
        'global_step': checkpoint['global_step'],
        'epoch_step': checkpoint['epoch_step'],
        'best_val_loss': checkpoint['best_val_loss'],
    }
    return model, optimizer, lr_scheduler, state

In [5]:
ROOT_FOLDER_DIR = "../data/brain-tumor-mri-deduplicated/"
OUT_DIR = Path().resolve().parent / 'models/vgg16/experiments/'
LEARNING_RATE = 1e-4
BATCH_SIZE = 4
NUM_EPOCHS = 2
CKPT_FREQ = 100  # autosaved checkpoint per 100 global batch steps

EXPERIMENT_NAME = "test"

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transforms = get_train_transforms()
val_transforms = get_val_transforms()

train_dataset = BrainMRI(ROOT_FOLDER_DIR, split='train', transform=train_transforms)
val_dataset = BrainMRI(ROOT_FOLDER_DIR, split='val', transform=val_transforms)

print(
    f"Train dataset loaded with {len(train_dataset)} samples "
    f"\nVal dataset loaded with {len(val_dataset)} samples"
)

train_dataloader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

Train dataset loaded with 4530 samples 
Val dataset loaded with 502 samples


In [8]:
model = VGG16(num_classes=4)
model= model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, fused=True)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer = optimizer, T_max=100, eta_min=LEARNING_RATE * 1e-2)
criterion = torch.nn.CrossEntropyLoss()

state = {
    "epoch": 0,
    "global_step": 0,
    "epoch_step": 0,
    "best_val_loss": float('inf'),
}

if EXPERIMENT_NAME is not None and (OUT_DIR / EXPERIMENT_NAME).is_dir():
    OUT_DIR = OUT_DIR / EXPERIMENT_NAME
    print(OUT_DIR)
    model, optimizer, lr_scheduler, state = load_checkpoint(model, optimizer, lr_scheduler, OUT_DIR, device)
    print(f"Checkpoint loaded, resuming at global step: {state['global_step']}")
elif EXPERIMENT_NAME is not None and not (OUT_DIR / EXPERIMENT_NAME).is_dir():
    try:
        OUT_DIR = OUT_DIR / EXPERIMENT_NAME
        os.makedirs(OUT_DIR, exist_ok=True)
        print(f"Starting new experiment, saving results at {OUT_DIR}")
    except Exception as e:
        print(f"Something went wrong {e}")

Starting new experiment, saving results at C:\Users\victo\Documents\VAULT\Machine Learning\homemade_machine_learning (maintained)\models\vgg16\experiments\test\test


In [79]:
for state["epoch"] in range(NUM_EPOCHS):
    print(f"Begining epoch {state['epoch']}")

    progress_bar = tqdm.tqdm(
        enumerate(train_dataloader),
        total=len(train_dataloader),
        desc=f"Step {state['epoch_step']}",
        initial=state["epoch_step"]
    )

    for epoch_step, batch in progress_bar:
        if epoch_step < state["epoch_step"]:
            continue

        input, label = batch[0], batch[1]
        input, label = input.to(device), label.to(device)
        output = model(input)
        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state["epoch_step"] += 1
        state["global_step"] += 1

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "global_step": state['global_step']
        })

        if state["global_step"] % CKPT_FREQ == 0:
            save_checkpoint(model, optimizer, lr_scheduler, state, OUT_DIR)

    state["epoch_step"] = 0

    running_val_loss = 0.
    with torch.no_grad():
        for epoch_step, batch in enumerate(val_dataloader):
            input_val, label_val = batch[0], batch[1]
            input_val, label_val = input_val.to(device), label_val.to(device)

            output_val = model(input_val)
            loss_val = criterion(output_val, label_val)

            running_val_loss += loss_val

    current_val_loss = running_val_loss / len(val_dataloader)
    print(f"Epoch {state['epoch']} | Validation loss {current_val_loss:.4f}")
    if current_val_loss < state["best_val_loss"]:
        state["best_val_loss"] = current_val_loss
        save_checkpoint(model, optimizer, lr_scheduler, state, OUT_DIR)

    lr_scheduler.step()

c:\Users\victo\anaconda3\envs\deepsearch\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Begining epoch 0


  0%|          | 0/1133 [1:46:29<?, ?it/s]?it/s]


Step 0:   9%|▉         | 100/1133 [02:38<40:29,  2.35s/it, loss=1.3693, global_step=100]

Checkpoint saved at epoch 0


Step 0:  15%|█▌        | 171/1133 [04:28<25:12,  1.57s/it, loss=1.4507, global_step=171]


KeyboardInterrupt: 